# RTIH — Text-to-SQL demo (structured data)

**This is NOT RAG.** For structured data (ERP, invoices, tables), you don't embed and retrieve — the LLM **writes a SQL query**, the database runs it, and you get an **exact** answer. No embeddings, no vectors.

Same LLM as the RAG starter (OpenRouter). Different job: it generates SQL. Run the cells top to bottom.

### Step 0 — install (run once, ~30s)
Red dependency warnings from Colab's pre-installed packages are normal — ignore them.

In [ ]:
!pip -q install langchain-community langchain-openai sqlalchemy

### Step 0b — paste your API key (OpenRouter)

In [ ]:
import os, getpass
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key: ')

### Step 1 — build a tiny sample database (a project's invoices)
In real life this is your ERP / accounting database. Here we make a small SQLite table so it runs out of the box.

In [ ]:
import sqlite3
con = sqlite3.connect("project.db")
con.executescript("""
DROP TABLE IF EXISTS invoices;
CREATE TABLE invoices (
  id INTEGER PRIMARY KEY, vendor TEXT, description TEXT,
  amount INTEGER, due_date TEXT, status TEXT, days_overdue INTEGER
);
INSERT INTO invoices (vendor, description, amount, due_date, status, days_overdue) VALUES
  ('Otis', 'Tower A lift cars', 1850000, '2026-05-20', 'overdue', 22),
  ('Spark Electricals', 'Tower B electrical rework', 800000, '2026-05-25', 'overdue', 17),
  ('AquaSeal', 'Podium waterproofing', 1400000, '2026-05-15', 'overdue', 27),
  ('Spark Electricals', 'Floor 7 RCD installation', 120000, '2026-06-01', 'paid', 0),
  ('BuildRight', 'Tower A structural works', 5200000, '2026-04-30', 'paid', 0),
  ('AquaSeal', 'Basement tanking', 600000, '2026-06-05', 'pending', 0),
  ('Otis', 'Lift annual maintenance', 95000, '2026-05-10', 'overdue', 32),
  ('GreenScape', 'Podium landscaping', 750000, '2026-06-14', 'pending', 0);
""")
con.commit(); con.close()
print("Sample invoices table created.")

### Step 2 — connect the DB + the LLM (the LLM will write the SQL)

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI

db = SQLDatabase.from_uri("sqlite:///project.db")
llm = ChatOpenAI(model="anthropic/claude-haiku-4.5",
                 base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"], max_tokens=512)

SCHEMA = db.get_table_info()

def clean_sql(text):
    s = text.strip().replace("```sql", "").replace("```sqlite", "").replace("```", "")
    for p in ("SQLQuery:", "SQLite query:", "SQL:"):
        if p in s: s = s.split(p, 1)[1]
    return s.strip().rstrip(";").strip()

def ask(question):
    prompt = ("You are a SQLite expert. Given this database schema:\n"
              f"{SCHEMA}\n\n"
              "Write ONE SQLite query that answers the question. "
              "Return ONLY the SQL - no explanation, no markdown.\n\n"
              f"Question: {question}")
    sql = clean_sql(llm.invoke(prompt).content)
    print(f"\nQ:   {question}\nSQL: {sql}\nA:   {db.run(sql)}")

### Step 3 — ask in plain English
The LLM writes the SQL; the database returns exact numbers. Change these to your own questions.

In [ ]:
ask("What is the total amount still overdue?")
ask("Which vendor have we paid the most in total, and how much?")
ask("How many invoices are more than 25 days overdue?")

### The point
RAG retrieves **text** by meaning. **Text-to-SQL writes a query** and the database computes the **exact** answer — that's why it's right on numbers where RAG isn't. Match the data to the tool.